In [ ]:
# add parent to path
import sys
from pathlib import Path

sys.path.append(Path.cwd().parent.parent.as_posix())

from demo_and_analysis.plots.utils.wandb_utils import combine_histories, get_wandb_stats

# tpc-h bespoke storage lineage
run_lineages = {
    # "tpch": [
    #     "nhpul25g",  # plan storage
    #     "a2tlnfrk",  # basic impl
    #     "h1mv9o9r",  # optim
    # ],
    # "ceb": [
    #     "x527bk9j",  # plan storage
    #     "blqeh6i0",  # basic impl
    #     "5qumtphx",  # optim
    # ],
    # revision
    "tpch": ["jg4jjo9u", "86crnuc0", "3zdiw9ol", "tc2q4e36"],
    "ceb": ["5m2t5fjn", "iv5w7m07", "gava9bsh", "2h03uj74"],
}

# run_id = "k9xst454"  # tpch - bespoke storage
# run_id = "ggikqms8"  # ceb
# run_id = "761bg8oe"  # ceb - bespoke storage

combined_hist_dict = dict()
for name, run_ids in run_lineages.items():
    hists_list = []
    for run_id in run_ids:
        summary, hist, config = get_wandb_stats(
            run_id,
            skip_cache=False,  # set to True to skip cache and fetch from W&B API
            wandb_run_cache_path=Path("/mnt/labstore/bespoke_olap/wandb_cache"),
        )
        hists_list.append(hist)

    combined_history = combine_histories(hists_list)

    # rewrite type if validation/external_call=True to validate_external
    combined_history["type"] = combined_history.apply(
        lambda row: (
            "validate_external"
            if row["validation/external_call"] == True
            else row["type"]
        ),
        axis=1,
    )

    combined_hist_dict[name] = combined_history

Loaded wandb data from cache: /mnt/labstore/bespoke_olap/wandb_cache/21f74321f09a12f51c273f3d499521892dba391dff5ab6eb673a44a7a001b0fa.pkl
Loaded wandb data from cache: /mnt/labstore/bespoke_olap/wandb_cache/532712d2907861c4bd7972ded9695a464e32cbe45a45dbabc4eab2e998ff5ba3.pkl
Loaded wandb data from cache: /mnt/labstore/bespoke_olap/wandb_cache/d9d6be482e0f01fd048c076a1ba8d75369053729bbb25c711294920a330b412f.pkl
Loaded wandb data from cache: /mnt/labstore/bespoke_olap/wandb_cache/55d752d2ee90cd93a69f9555f05f322483f3728feac2a1e47e6c09550c413d76.pkl
Combined history has 15130 rows ([5, 1544, 8310, 5271])
Loaded wandb data from cache: /mnt/labstore/bespoke_olap/wandb_cache/5089d5e1738d26656475a3b15f557634eef25359edcd8c0c3c6512187360cdd7.pkl
Loaded wandb data from cache: /mnt/labstore/bespoke_olap/wandb_cache/993b82494ba49cdcc6d8deda5d1dd7b8975b1c3e7e76964eb8a68d32270fc23a.pkl
Loaded wandb data from cache: /mnt/labstore/bespoke_olap/wandb_cache/598a2f88a24ed66aec7b4ee9e514a79d78b3cf8bd099856

# Compile Tool Stats

In [8]:
def get_compile_stats(combined_history, verbose: bool = False):
    # count compile invocations
    # considering only internal managed compile calls for now, as external calls are often just benchmarking

    filtered_hist = combined_history[
        combined_history["type"].isin(["compile", "validate"])
    ]

    filtered_tool_counts = filtered_hist["type"].value_counts().to_dict()

    compile_count = filtered_tool_counts.get("compile", 0)
    validate_count = filtered_tool_counts.get("validate", 0)

    # count errors
    compile_err_col = "compile/compile_error"
    if compile_err_col in filtered_hist.columns:
        compile_error_ctr = (
            filtered_hist[compile_err_col].value_counts().to_dict().get(True, 0)
        )
    else:
        compile_error_ctr = 0
    validate_compile_error_ctr = (
        filtered_hist["validation/compile_error"].value_counts().to_dict().get(True, 0)
    )

    # totals
    total_count = compile_count + validate_count
    total_errors = compile_error_ctr + validate_compile_error_ctr

    if verbose:
        print(
            f"Total compile-related tool invocations: {total_count}, with {total_errors} errors (error rate: {total_errors / total_count * 100 if total_count > 0 else 0:.2f}%) - compile only: {compile_count} (errors: {compile_error_ctr})."
        )

    return {
        "count": total_count,
        "error_rate": total_errors / total_count if total_count > 0 else 0,
    }


get_compile_stats(combined_hist_dict["tpch"], verbose=True)

Total compile-related tool invocations: 2392, with 56 errors (error rate: 2.34%) - compile only: 401 (errors: 0).


{'count': 2392, 'error_rate': 0.023411371237458192}

# Shell Tool Stats

In [9]:
import shlex


def _cmd_head(cmd: str) -> str:
    if not isinstance(cmd, str):
        return ""
    cmd = cmd.strip()
    if not cmd:
        return ""
    try:
        parts = shlex.split(cmd)
    except Exception:
        parts = cmd.split()
    return parts[0].lower() if parts else ""


def _has_any_token(cmd: str, tokens: set[str]) -> bool:
    if not isinstance(cmd, str):
        return False
    try:
        parts = [p.lower() for p in shlex.split(cmd)]
    except Exception:
        parts = cmd.lower().split()
    return any(t in parts for t in tokens)


def get_shell_tool_stats(combined_history, verbose: bool = False):
    # get shell tool stats
    shell_calls = combined_history[combined_history["type"] == "shell"]

    num_shell_tool_calls = shell_calls.shape[0]

    # flatten shell cmd list: shell/commands
    shell_cmds = shell_calls["shell/commands"].explode()
    # classify shell cmds by intent (more semantic than raw startswith)
    classes = {
        "filesystem (ls, find, ...)": lambda cmd: (
            _cmd_head(cmd) in {"ls", "find", "tree", "pwd", "stat", "du", "file"}
        ),
        "file reading (cat, ...)": lambda cmd: (
            _cmd_head(cmd) in {"cat", "head", "tail", "less", "more", "bat"}
        ),
        "text processing/search (grep, ...)": lambda cmd: (
            _cmd_head(cmd)
            in {"grep", "rg", "ag", "sed", "awk", "cut", "sort", "uniq", "wc", "nl"}
        ),
        "code execution (python, ...)": lambda cmd: (
            _cmd_head(cmd)
            in {"python", "python3", "ipython", "bash", "sh", "zsh", "make", "cmake"}
        ),
        "parquet analysis": lambda cmd: (
            _cmd_head(cmd) in {"parquet-tools", "parquet-dump-schema", "jq"}
        ),
        "environment/cmd lookup (which, ...)": lambda cmd: (
            _cmd_head(cmd) in {"which", "whereis", "type", "env", "printenv", "git"}
        ),
        "file mutation (cp, mv, ...)": lambda cmd: (
            _cmd_head(cmd)
            in {"cp", "mv", "rm", "mkdir", "rmdir", "touch", "chmod", "chown"}
        ),
        "output/formatting (echo, printf)": lambda cmd: (
            _cmd_head(cmd) in {"echo", "printf"}
        ),
        "pipeline-heavy": lambda cmd: _has_any_token(cmd, {"|", "xargs"}),
    }

    shell_cmd_classes = shell_cmds.apply(
        lambda cmd: next((cls for cls, fn in classes.items() if fn(cmd)), cmd)
    )

    if verbose:
        print(shell_cmd_classes.value_counts().to_dict())

    return {
        "count": num_shell_tool_calls,
        "shell_cmd_classes": shell_cmd_classes.value_counts().to_dict(),
    }


get_shell_tool_stats(combined_hist_dict["tpch"], verbose=True)

{'file reading (cat, ...)': 1684, 'text processing/search (grep, ...)': 1130, 'filesystem (ls, find, ...)': 640, 'pipeline-heavy': 203, 'code execution (python, ...)': 117, 'file mutation (cp, mv, ...)': 59, 'output/formatting (echo, printf)': 10, 'nproc': 7, 'parquet analysis': 5, 'environment/cmd lookup (which, ...)': 4, 'md5sum /home/jwehrstein/bespoke_olap/output/build/libbuilder.so /home/jwehrstein/bespoke_olap/output/build/.reload/libbuilder.so.1323651.3.so /home/jwehrstein/bespoke_olap/output/build/.reload/libbuilder.so.1325330.2.so': 1, 'md5sum /home/jwehrstein/bespoke_olap/output/build/libloader.so 2>/dev/null; ls -la /home/jwehrstein/bespoke_olap/output/build/libloader.so 2>/dev/null': 1, 'md5sum /home/jwehrstein/bespoke_olap/output/build/libloader.so /home/jwehrstein/bespoke_olap/output/build/.reload/libloader.so.1325831.2.so': 1, "OPENBLAS_NUM_THREADS=1 python3 << 'EOF'\nimport pyarrow.parquet as pq\nmeta = pq.read_metadata('/mnt/labstore/bespoke_olap/tpch_parquet/sf2/linei

{'count': 3888,
 'shell_cmd_classes': {'file reading (cat, ...)': 1684,
  'text processing/search (grep, ...)': 1130,
  'filesystem (ls, find, ...)': 640,
  'pipeline-heavy': 203,
  'code execution (python, ...)': 117,
  'file mutation (cp, mv, ...)': 59,
  'output/formatting (echo, printf)': 10,
  'nproc': 7,
  'parquet analysis': 5,
  'environment/cmd lookup (which, ...)': 4,
  'md5sum /home/jwehrstein/bespoke_olap/output/build/libbuilder.so /home/jwehrstein/bespoke_olap/output/build/.reload/libbuilder.so.1323651.3.so /home/jwehrstein/bespoke_olap/output/build/.reload/libbuilder.so.1325330.2.so': 1,
  'md5sum /home/jwehrstein/bespoke_olap/output/build/libloader.so 2>/dev/null; ls -la /home/jwehrstein/bespoke_olap/output/build/libloader.so 2>/dev/null': 1,
  'md5sum /home/jwehrstein/bespoke_olap/output/build/libloader.so /home/jwehrstein/bespoke_olap/output/build/.reload/libloader.so.1325831.2.so': 1,
  "OPENBLAS_NUM_THREADS=1 python3 << 'EOF'\nimport pyarrow.parquet as pq\nmeta = pq.

# Apply Patch Stats

In [10]:
def get_apply_patch_stats(combined_history, verbose: bool = False):
    filtered_hist = combined_history[(combined_history["type"] == "apply_patch")]

    num_apply_patch_calls = len(filtered_hist)

    # compute avg between added and deleted loc count
    diff_size = (
        filtered_hist[["apply_patch/added_loc_count", "apply_patch/deleted_loc_count"]]
        .fillna(0)
        .max(axis=1)
    )

    # compute avg diff size
    avg_diff_size = float(diff_size.mean())

    if verbose:
        print(
            f"Total apply_patch_tool calls: {num_apply_patch_calls}, with average diff size (added or deleted LOC): {avg_diff_size:.2f}."
        )

    return {
        "count": num_apply_patch_calls,
        "avg_diff_size": avg_diff_size,
    }


get_apply_patch_stats(combined_hist_dict["tpch"], verbose=True)

Total apply_patch_tool calls: 1000, with average diff size (added or deleted LOC): 2.54.


{'count': 1000, 'avg_diff_size': 2.538}

# Validate Call

In [11]:
def get_validate_stats(combined_history, verbose: bool = False):
    # count compile invocations
    # considering only internal managed compile calls for now, as external calls are often just benchmarking

    type_mask = combined_history["type"].isin(["validate", "validate_external"])
    no_compile_error_mask = (
        combined_history["validation/compile_error"].fillna(False).eq(False)
    )  # keep rows where compile_error is False or NaN

    filtered_hist = combined_history[type_mask & no_compile_error_mask]

    validate_count = filtered_hist[filtered_hist["type"] == "validate"].shape[0]
    validate_external_count = filtered_hist[
        filtered_hist["type"] == "validate_external"
    ].shape[0]

    # count errors
    incorrect_elems = filtered_hist[filtered_hist["validation/correct"] == False]
    validate_errors = incorrect_elems[incorrect_elems["type"] == "validate"].shape[0]
    validate_errors_external = incorrect_elems[
        incorrect_elems["type"] == "validate_external"
    ].shape[0]

    # compute error rate internal
    error_rate_internal = validate_errors / validate_count if validate_count > 0 else 0
    error_rate_external = (
        validate_errors_external / validate_external_count
        if validate_external_count > 0
        else 0
    )

    return {
        "count": validate_count,
        "error_rate_internal": error_rate_internal,
        "validate_external_count": validate_external_count,
        "error_rate_external": error_rate_external,
    }


get_validate_stats(combined_hist_dict["tpch"], verbose=True)

{'count': 1935,
 'error_rate_internal': 0.03204134366925065,
 'validate_external_count': 261,
 'error_rate_external': 0.007662835249042145}

# Assemble Output

In [13]:
rows = []
shell_types = dict()

TOOL_TYPES = {
    "shell",
    "apply_patch",
    "compile",
    "validate",
    "validate_external",
}

for name, combined_history in combined_hist_dict.items():
    compile_stats = get_compile_stats(combined_history=combined_history)
    shell_stats = get_shell_tool_stats(combined_history=combined_history)
    apply_patch_stats = get_apply_patch_stats(combined_history=combined_history)
    validate_stats = get_validate_stats(combined_history=combined_history)

    tool_type_mask = combined_history["type"].isin(TOOL_TYPES)
    tool_count = int(tool_type_mask.sum())
    llm_count = int((~tool_type_mask).sum())
    total_interactions = int(len(combined_history))
    llm_share = llm_count / total_interactions if total_interactions > 0 else 0

    row = [
        name,
        f"{total_interactions}",
        f"{llm_count} ({llm_share * 100:.2f}%)",
        f"{shell_stats['count']}",
        f"{apply_patch_stats['count']} (avg diff size: {apply_patch_stats['avg_diff_size']:.2f} LOC)",
        f"{compile_stats['count']} (errors: {compile_stats['error_rate'] * 100:.2f}%)",
        f"{validate_stats['count']} (errors: {validate_stats['error_rate_internal'] * 100:.2f}%)\n + {validate_stats['validate_external_count']} framework invocations",
        f"{tool_count}",
    ]
    rows.append(row)

    shell_types[name] = shell_stats["shell_cmd_classes"]

# print tabulate
from collections import defaultdict

from tabulate import tabulate

print(
    tabulate(
        rows,
        headers=[
            "Name",
            "Total",
            "LLM Interactions",
            "Shell",
            "Apply Patch",
            "Compile",
            "Validate",
            "Tool Calls",
        ],
    )
)


# print shell tool types distribution
total_type_count = defaultdict(int)
for name, shell_cmd_classes in shell_types.items():
    for cls, count in shell_cmd_classes.items():
        total_type_count[cls] += count

# print relative
total_calls = sum(total_type_count.values())
dist_rows = [
    [cls[:100], count, f"{(count / total_calls * 100) if total_calls else 0:.2f}%"]
    for cls, count in total_type_count.items()
]

print(
    tabulate(
        dist_rows,
        headers=["Shell Command Type", "Count", "Share"],
    )
)

# print llm interaction type distribution
total_llm_type_count = defaultdict(int)
for combined_history in combined_hist_dict.values():
    llm_hist = combined_history[~combined_history["type"].isin(TOOL_TYPES)]
    for llm_type, count in llm_hist["type"].value_counts().to_dict().items():
        total_llm_type_count[llm_type] += int(count)

llm_total_calls = sum(total_llm_type_count.values())
llm_dist_rows = [
    [
        llm_type,
        count,
        f"{(count / llm_total_calls * 100) if llm_total_calls else 0:.2f}%",
    ]
    for llm_type, count in total_llm_type_count.items()
]

print(
    tabulate(
        llm_dist_rows,
        headers=["LLM Interaction Type", "Count", "Share"],
    )
)

Name      Total  LLM Interactions      Shell  Apply Patch                     Compile               Validate                        Tool Calls
------  -------  ------------------  -------  ------------------------------  --------------------  ----------------------------  ------------
tpch      15130  7589 (50.16%)          3888  1000 (avg diff size: 2.54 LOC)  2392 (errors: 2.34%)  1935 (errors: 3.20%)                  7541
                                                                                                     + 261 framework invocations
ceb       14669  7350 (50.11%)          3695  1115 (avg diff size: 4.86 LOC)  2307 (errors: 2.77%)  1804 (errors: 8.65%)                  7319
                                                                                                     + 202 framework invocations
Shell Command Type                                                                                      Count  Share
-----------------------------------------------------